#**TP 1 - ESTATÍSTICA APLICADA E ANÁLISE EXPLORATÓRIA DE DADOS**

#CÓDIGO PARA GERAÇÃO DOS DATASETS



In [37]:
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta

In [38]:
rng = np.random.default_rng(42)
n = 4000
start_date = datetime(2026, 1, 5)

ticket_ids = [f"TCK-{i:05d}" for i in range(1, n + 1)]
cliente_ids = [f"CLI-{i:03d}" for i in rng.integers(1, 181, n)]
dias_offset = np.sort(rng.integers(0, 180, n))
datas = [(start_date + timedelta(days=int(d))).strftime("%Y-%m-%d") for d in dias_offset]
canais = rng.choice(["email", "chat", "telefone"], size=n, p=[0.45, 0.35, 0.20])
categorias = rng.choice(["bug", "duvida_de_uso", "cobranca", "elogio"], size=n, p=[0.35, 0.30, 0.25, 0.10])
prioridades = rng.choice(["baixa", "media", "alta", "critica"], size=n, p=[0.30, 0.35, 0.25, 0.10])
tempo_resposta = np.clip(rng.gamma(shape=2.0, scale=2.2, size=n), 0.2, 48)
tempo_resposta[canais == "chat"] *= 0.6
tempo_resolucao = np.clip(1.8 * tempo_resposta + 3 + rng.normal(0, 1.5, n), 0.5, 120)
satisfacao = 5 - (tempo_resposta / 6) + rng.normal(0, 0.6, n)
satisfacao[canais == "chat"] -= 0.6
satisfacao = np.clip(np.round(satisfacao), 1, 5)

df = pd.DataFrame({
    "ticket_id": ticket_ids,
    "cliente_id": cliente_ids,
    "data_abertura": datas,
    "canal": canais,
    "categoria": categorias,
    "prioridade": prioridades,
    "tempo_resposta_horas": tempo_resposta,
    "tempo_resolucao_horas": tempo_resolucao,
    "satisfacao_cliente": satisfacao,
})

# valores ausentes (como em uma exportação real)
df.loc[rng.choice(n, size=int(n * 0.08), replace=False), "tempo_resposta_horas"] = np.nan
df.loc[rng.choice(n, size=int(n * 0.10), replace=False), "satisfacao_cliente"] = np.nan

# tempo_resposta_horas exportado como texto, com vírgula decimal
def fmt_virgula(v):
    return "" if pd.isna(v) else f"{v:.2f}".replace(".", ",")

df["tempo_resposta_horas"] = df["tempo_resposta_horas"].apply(fmt_virgula)

# duplicatas de reenvio, anexadas ao final (não afetam a ordem cronológica das primeiras n linhas)
duplicatas = df.sample(frac=0.02, random_state=42)
df_final = pd.concat([df, duplicatas], ignore_index=True)
df_final.to_csv("tickets_suporte.csv", index=False)

In [39]:
# planos_clientes.xlsx
clientes_unicos = sorted(set(cliente_ids))
planos = pd.DataFrame({
    "cliente_id": clientes_unicos,
    "valor_plano": rng.choice([99.0, 199.0, 349.0, 599.0], size=len(clientes_unicos), p=[0.4, 0.3, 0.2, 0.1]),
    "segmento": rng.choice(["starter", "growth", "enterprise"], size=len(clientes_unicos), p=[0.5, 0.35, 0.15]),
})
planos.to_excel("planos_clientes.xlsx", index=False)

In [40]:
# chatbot_triagem.json (apenas tickets abertos pelo canal chat)
chat_ids = df.loc[df["canal"] == "chat", "ticket_id"].tolist()
triagem = [
    {
        "ticket_id": tid,
        "intencao_triagem": str(rng.choice(["duvida", "reclamacao", "solicitacao", "elogio"])),
        "confianca_bot": round(float(rng.uniform(0.55, 0.98)), 2),
    }
    for tid in chat_ids
]

with open("chatbot_triagem.json", "w", encoding="utf-8") as f:
    json.dump(triagem, f, ensure_ascii=False, indent=2)

#TP1 - Respostas

###Imports

In [41]:
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta

##**EXERCÍCIO 1**

### 1.1 Carregue o arquivo tickets_suporte.csv com pandas.read_csv.

In [42]:
df_tickets = pd.read_csv("tickets_suporte.csv")
df_tickets

,ticket_id,cliente_id,data_abertura,canal,categoria,prioridade,tempo_resposta_horas,tempo_resolucao_horas,satisfacao_cliente
0,TCK-00001,CLI-017,2026-01-05,chat,duvida_de_uso,media,"1,54",6.833510,4.0
1,TCK-00002,CLI-140,2026-01-05,email,cobranca,media,"2,05",6.664003,5.0
2,TCK-00003,CLI-118,2026-01-05,telefone,bug,baixa,"2,04",6.184528,4.0
3,TCK-00004,CLI-079,2026-01-05,email,bug,critica,"5,19",10.471215,3.0
4,TCK-00005,CLI-078,2026-01-05,chat,cobranca,baixa,"0,63",3.049426,4.0
...,...,...,...,...,...,...,...,...,...
4075,TCK-03149,CLI-179,2026-05-24,chat,bug,media,"3,26",8.401522,4.0
4076,TCK-03552,CLI-094,2026-06-12,chat,cobranca,media,"1,08",6.071202,5.0
4077,TCK-01445,CLI-026,2026-03-07,telefone,cobranca,media,"6,90",14.740413,4.0
4078,TCK-00394,CLI-108,2026-01-21,chat,cobranca,baixa,"1,03",4.428727,4.0


### 1.2 Inspecione o DataFrame resultante com shape, dtypes e info(), registrando o número de linhas, colunas e o tipo inferido para cada uma.

In [43]:
print("shape:")
print(df_tickets.shape)

print("\ndtypes:")
print(df_tickets.dtypes)

print("\ninfo:")
df_tickets.info()

shape:
(4080, 9)

dtypes:
ticket_id                 object
cliente_id                object
data_abertura             object
canal                     object
categoria                 object
prioridade                object
tempo_resposta_horas      object
tempo_resolucao_horas    float64
satisfacao_cliente       float64
dtype: object

info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4080 entries, 0 to 4079
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ticket_id              4080 non-null   object 
 1   cliente_id             4080 non-null   object 
 2   data_abertura          4080 non-null   object 
 3   canal                  4080 non-null   object 
 4   categoria              4080 non-null   object 
 5   prioridade             4080 non-null   object 
 6   tempo_resposta_horas   3751 non-null   object 
 7   tempo_resolucao_horas  4080 non-null   float64
 8   satisfacao_cliente     36

### 1.3 Escreva, em uma célula de markdown do notebook, três observações sobre o dataset com base nessa inspeção (ex.: colunas com tipo object que deveriam ser numéricas ou datas).

A coluna *tempo_resposta_horas* foi inferida como *object*, que pelo nome e pela amostra análisada deveria ter sido inferida como numérica, sinalizando será nescessário realizar um trabalho de conversão e validação de tipos antes de qualquer cálculo estatístico

A coluna *data_abertura* também foi inferida como *object* mesmo representando datas, o que irá impedir a ordenação cronológica ou cálculo de intervalos de tempo sem uma conversão prévia para *datetime64*

As colunas *tempo_resposta_horas* e *satisfacao_cliente* têm menos valores não nulos que o total de linhas, evidenciando que há dados ausentes que precisarão de tratamento antes de qualquer agregação

##**EXERCÍCIO 2**

### 2.1 Importe o arquivo planos_clientes.xlsx com pandas.read_excel.

In [44]:
df_planos = pd.read_excel("planos_clientes.xlsx")
df_planos

,cliente_id,valor_plano,segmento
0,CLI-001,99,growth
1,CLI-002,199,growth
2,CLI-003,99,starter
3,CLI-004,599,growth
4,CLI-005,349,growth
...,...,...,...
175,CLI-176,199,starter
176,CLI-177,99,starter
177,CLI-178,349,starter
178,CLI-179,199,growth


### 2.2 Importe o arquivo chatbot_triagem.json com pandas.read_json.

In [45]:
df_triagem = pd.read_json("chatbot_triagem.json")
df_triagem

,ticket_id,intencao_triagem,confianca_bot
0,TCK-00001,reclamacao,0.92
1,TCK-00005,solicitacao,0.90
2,TCK-00007,reclamacao,0.81
3,TCK-00010,reclamacao,0.74
4,TCK-00015,solicitacao,0.62
...,...,...,...
1392,TCK-03975,elogio,0.95
1393,TCK-03985,reclamacao,0.95
1394,TCK-03989,reclamacao,0.96
1395,TCK-03991,solicitacao,0.79


###2.3 Para cada uma das três fontes (CSV do Exercício 1, Excel, JSON), imprima as primeiras 5 linhas e o número de colunas, confirmando que a importação preservou os dados esperados.

In [46]:
print("tickets:")
print(df_tickets.head())
print(df_tickets.shape[1], "colunas\n")

print("planos:")
print(df_planos.head())
print(df_planos.shape[1], "colunas\n")

print("triagem:")
print(df_triagem.head())
print(df_triagem.shape[1], "colunas")

tickets:
   ticket_id cliente_id data_abertura     canal      categoria prioridade  \
0  TCK-00001    CLI-017    2026-01-05      chat  duvida_de_uso      media   
1  TCK-00002    CLI-140    2026-01-05     email       cobranca      media   
2  TCK-00003    CLI-118    2026-01-05  telefone            bug      baixa   
3  TCK-00004    CLI-079    2026-01-05     email            bug    critica   
4  TCK-00005    CLI-078    2026-01-05      chat       cobranca      baixa   

  tempo_resposta_horas  tempo_resolucao_horas  satisfacao_cliente  
0                 1,54               6.833510                 4.0  
1                 2,05               6.664003                 5.0  
2                 2,04               6.184528                 4.0  
3                 5,19              10.471215                 3.0  
4                 0,63               3.049426                 4.0  
9 colunas

planos:
  cliente_id  valor_plano segmento
0    CLI-001           99   growth
1    CLI-002          199   gro

##**EXERCÍCIO 3**

### 3.1 Utilize dtypes para listar o tipo atual de data_abertura, prioridade e tempo_resposta_horas.

In [47]:
print(df_tickets[["data_abertura", "prioridade", "tempo_resposta_horas"]].dtypes)

data_abertura           object
prioridade              object
tempo_resposta_horas    object
dtype: object


As três colunas estam aparecendo como *object*, por estarem como *string*, apesar de conceitualmente representem uma data, uma categoria e um número.

### 3.2 Converta data_abertura para datetime64 e tempo_resposta_horas para float, ambos com astype (após tratar o separador decimal quando necessário).

In [48]:
df_tickets["data_abertura"] = df_tickets["data_abertura"].astype("datetime64[ns]")

df_tickets["tempo_resposta_horas"] = (
    df_tickets["tempo_resposta_horas"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print(df_tickets[["data_abertura", "prioridade", "tempo_resposta_horas"]].dtypes)

data_abertura           datetime64[ns]
prioridade                      object
tempo_resposta_horas           float64
dtype: object


Antes de realizar a conversão para *float* no tempo resposta, foi nescessário realizar a troca da vírgula pelo ponto com o *replace*, porque o *astype* não sabe interpretar vírgula como separador decimal.

### 3.3  Explique, em uma célula de markdown, por que manter prioridade como object em vez de convertê-la seria uma decisão ruim para um dataset com centenas de milhares de linhas.

O tipo *object* armazena cada valor como uma *string* independente, então mesmo repetindo apenas quatro valores possíveis (baixa, media, alta, critica), o *pandas* guarda o texto completo em cada uma das linhas. O tipo category, em vez disso, mantém um pequeno dicionário com os valores únicos e guarda, em cada linha, só um código inteiro que aponta para esse dicionário.
Como o número de valores únicos de prioridade não cresce junto com o tamanho do *dataset*, essa diferença de armazenamento fica cada vez mais vantajosa quanto mais linhas existirem, chegando a economizar muita memória em um *dataset* com centenas de milhares de registros. Além do ganho de memória, operações como groupby, ordenação e comparação também ficam mais rápidas com o *category*, porque comparar códigos inteiros é mais barato que comparar *strings* repetidamente.

##**EXERCÍCIO 4**

### 4.1 Utilize loc para selecionar apenas as colunas ticket_id, categoria e tempo_resposta_horas de todo o dataset.


In [49]:
subset_colunas = df_tickets.loc[:, ["ticket_id", "categoria", "tempo_resposta_horas"]]
subset_colunas

,ticket_id,categoria,tempo_resposta_horas
0,TCK-00001,duvida_de_uso,1.54
1,TCK-00002,cobranca,2.05
2,TCK-00003,bug,2.04
3,TCK-00004,bug,5.19
4,TCK-00005,cobranca,0.63
...,...,...,...
4075,TCK-03149,bug,3.26
4076,TCK-03552,cobranca,1.08
4077,TCK-01445,cobranca,6.90
4078,TCK-00394,cobranca,1.03


### 4.2 Utilize iloc para selecionar as linhas correspondentes às posições 500 a 800 (a segunda semana de dados, segundo a ordem de carregamento) desse subconjunto de colunas.

In [50]:
segunda_semana = subset_colunas.iloc[500:800]

print("shape:")
print(segunda_semana.shape)

print("\nhead:")
print(segunda_semana.head())

print("\ntail:")
print(segunda_semana.tail())

shape:
(300, 3)

head:
     ticket_id      categoria  tempo_resposta_horas
500  TCK-00501            bug                   NaN
501  TCK-00502       cobranca                  4.20
502  TCK-00503            bug                  1.39
503  TCK-00504  duvida_de_uso                  2.13
504  TCK-00505       cobranca                   NaN

tail:
     ticket_id      categoria  tempo_resposta_horas
795  TCK-00796  duvida_de_uso                  4.61
796  TCK-00797  duvida_de_uso                  2.85
797  TCK-00798  duvida_de_uso                  5.09
798  TCK-00799            bug                  0.42
799  TCK-00800  duvida_de_uso                  1.86


### 4.3 Compare em uma frase quando você usaria loc em vez de iloc em um pipeline real.

O *loc* deve ser usado para selecionar lógicamente, por rótulo, seja o nome de uma coluna, um valor de índice ou uma condição booleana, e o *iloc* deve ser usado quando preciso de uma posição numérica fixa independente dos rótulos, como quando precido de exatamente as linhas na posição 500 até a 799 do *DataFrame*, não importa qual seja o valor do índice naquelas linhas.

##**EXERCÍCIO 5**

### 5.1 Filtre o DataFrame de tickets para prioridade == "alta" e canal == "email" usando notação booleana composta com &.

In [51]:
filtro_booleano = df_tickets[(df_tickets["prioridade"] == "alta") & (df_tickets["canal"] == "email")]
filtro_booleano

,ticket_id,cliente_id,data_abertura,canal,categoria,prioridade,tempo_resposta_horas,tempo_resolucao_horas,satisfacao_cliente
8,TCK-00009,CLI-037,2026-01-05,email,cobranca,alta,6.75,15.088345,5.0
35,TCK-00036,CLI-012,2026-01-06,email,cobranca,alta,1.85,6.961171,5.0
38,TCK-00039,CLI-050,2026-01-06,email,bug,alta,1.64,4.030364,5.0
45,TCK-00046,CLI-175,2026-01-07,email,bug,alta,2.07,8.682418,5.0
62,TCK-00063,CLI-066,2026-01-07,email,duvida_de_uso,alta,NaN,5.180166,5.0
...,...,...,...,...,...,...,...,...,...
4052,TCK-00315,CLI-044,2026-01-18,email,bug,alta,4.74,12.016189,3.0
4063,TCK-03380,CLI-072,2026-06-04,email,bug,alta,4.87,12.915356,5.0
4067,TCK-00606,CLI-097,2026-01-30,email,duvida_de_uso,alta,2.83,8.623705,3.0
4073,TCK-00009,CLI-037,2026-01-05,email,cobranca,alta,6.75,15.088345,5.0


### 5.2 Reescreva o mesmo filtro usando o método query().

In [52]:
filtro_query = df_tickets.query("prioridade == 'alta' and canal == 'email'")
filtro_query

,ticket_id,cliente_id,data_abertura,canal,categoria,prioridade,tempo_resposta_horas,tempo_resolucao_horas,satisfacao_cliente
8,TCK-00009,CLI-037,2026-01-05,email,cobranca,alta,6.75,15.088345,5.0
35,TCK-00036,CLI-012,2026-01-06,email,cobranca,alta,1.85,6.961171,5.0
38,TCK-00039,CLI-050,2026-01-06,email,bug,alta,1.64,4.030364,5.0
45,TCK-00046,CLI-175,2026-01-07,email,bug,alta,2.07,8.682418,5.0
62,TCK-00063,CLI-066,2026-01-07,email,duvida_de_uso,alta,NaN,5.180166,5.0
...,...,...,...,...,...,...,...,...,...
4052,TCK-00315,CLI-044,2026-01-18,email,bug,alta,4.74,12.016189,3.0
4063,TCK-03380,CLI-072,2026-06-04,email,bug,alta,4.87,12.915356,5.0
4067,TCK-00606,CLI-097,2026-01-30,email,duvida_de_uso,alta,2.83,8.623705,3.0
4073,TCK-00009,CLI-037,2026-01-05,email,cobranca,alta,6.75,15.088345,5.0


### 5.3 Confirme que as duas abordagens retornam o mesmo número de linhas e explique, em uma frase, uma vantagem prática de query() sobre a notação booleana composta.

In [53]:
print("Filtro por booleano:")
print(filtro_booleano.shape[0])

print("\nFiltro por query:")
print(filtro_query.shape[0])

print("\nSão iguais?")
print(filtro_booleano.shape[0] == filtro_query.shape[0])

Filtro por booleano:
494

Filtro por query:
494

São iguais?
True


Ambas as abordagens retornam 494 linhas cada, confirmando resultados idênticos, porém, uma vantagem prática de *query()* é a legibilidade em filtros com várias condições, já que não é preciso repetir o nome do *DataFrame* a cada comparação nem envolver cada condição em parênteses para respeitar a precedência de operadores.

##**EXERCÍCIO 6**

### 6.1 Identifique quantos valores ausentes existem em cada coluna do dataset de tickets.

In [54]:
print(df_tickets.isna().sum())

ticket_id                  0
cliente_id                 0
data_abertura              0
canal                      0
categoria                  0
prioridade                 0
tempo_resposta_horas     329
tempo_resolucao_horas      0
satisfacao_cliente       407
dtype: int64


A coluna *tempo_resposta_horas* tem 329 valores *nulos* e *satisfacao_cliente* tem 407 nulos, as demais colunas estão completas

### 6.2 Trate tempo_resposta_horas ausente com uma estratégia de preenchimento justificada (por exemplo, mediana por categoria) usando fillna.

In [55]:
mediana_por_categoria = df_tickets.groupby("categoria")["tempo_resposta_horas"].transform("median")
df_tickets["tempo_resposta_horas"] = df_tickets["tempo_resposta_horas"].fillna(mediana_por_categoria)

print("Medianas por categoria:")
print(mediana_por_categoria.unique())

print("\nTotal de valores nulos em tempo_resposta_horas:")
print(df_tickets["tempo_resposta_horas"].isna().sum())

Medianas por categoria:
[3.025 3.21  3.12  3.125]

Total de valores nulos em tempo_resposta_horas:
0


Foi adotada a mediana por categoria porque a mediana é mais robusta que a média, já que o tempo de resposta tem cauda longa.

### 6.3 Remova, com dropna, os registros em que ticket_id esteja ausente (chave primária não pode faltar).

In [56]:
df_tickets = df_tickets.dropna(subset=["ticket_id"])
print(df_tickets.shape[0])

4080


##**EXERCÍCIO 7**

### 7.1 Identifique quantas linhas duplicadas existem no dataset de tickets, considerando ticket_id como identificador de duplicidade.


In [58]:
duplicatas_ticket = df_tickets.duplicated(subset=["ticket_id"]).sum()
duplicatas_ticket

np.int64(80)

### 7.2 Remova as duplicatas com drop_duplicates, mantendo o primeiro registro de cada ticket_id.

In [59]:
linhas_antes = df_tickets.shape[0]

print("Linhas antes:")
print(linhas_antes)

df_tickets = df_tickets.drop_duplicates(subset=["ticket_id"], keep="first")
linhas_depois = df_tickets.shape[0]

print("\nLinhas depois:")
print(linhas_depois)

Linhas antes:
4080

Linhas depois:
4000


### 7.3 Reporte, em uma célula de markdown, quantas linhas foram removidas e o percentual que isso representa do dataset original.

In [60]:
removidas = linhas_antes - linhas_depois
percentual = removidas / linhas_antes * 100
print(removidas, f"{percentual:.2f}%")

80 1.96%


Foram removidas 80 linhas, o que representa 1,96% do dataset original de 4080 registros, restando 4000 tickets únicos.

##**EXERCÍCIO 8**

### 8.1 Combine o DataFrame de tickets com o DataFrame de planos de clientes usando pd.merge, unindo pelo identificador de cliente comum às duas fontes.

In [62]:
df_consolidado = pd.merge(df_tickets, df_planos, on="cliente_id", how="left")

print("shape:")
print(df_consolidado.shape)

print("\nhead:")
print(df_consolidado.head())

print("\ntail:")
print(df_consolidado.tail())

shape:
(4000, 11)

head:
   ticket_id cliente_id data_abertura     canal      categoria prioridade  \
0  TCK-00001    CLI-017    2026-01-05      chat  duvida_de_uso      media   
1  TCK-00002    CLI-140    2026-01-05     email       cobranca      media   
2  TCK-00003    CLI-118    2026-01-05  telefone            bug      baixa   
3  TCK-00004    CLI-079    2026-01-05     email            bug    critica   
4  TCK-00005    CLI-078    2026-01-05      chat       cobranca      baixa   

   tempo_resposta_horas  tempo_resolucao_horas  satisfacao_cliente  \
0                  1.54               6.833510                 4.0   
1                  2.05               6.664003                 5.0   
2                  2.04               6.184528                 4.0   
3                  5.19              10.471215                 3.0   
4                  0.63               3.049426                 4.0   

   valor_plano    segmento  
0           99  enterprise  
1          199     starter  
2   

### 8.2 Combine o resultado com os registros de triagem do chatbot usando pd.concat, quando aplicável ao mesmo esquema de colunas, ou pd.merge quando as chaves permitirem.

In [ ]:
df_consolidado = pd.merge(df_consolidado, df_triagem, on="ticket_id", how="left")

print("shape:")
print(df_consolidado.shape)

print("\nhead:")
print(df_consolidado.head())

print("\ntail:")
print(df_consolidado.tail())

Como as duas tabelas não compartilham o mesmo esquema de colunas, um *concat* empilharia linhas com colunas incompatíveis e geraria uma explosão de valores nulos sem juntar informação nenhuma, enquanto o merge por *ticket_id* associa corretamente cada registro de triagem ao *ticket* correspondente.

###8.3 Confirme o número final de linhas e colunas do DataFrame consolidado.

In [65]:
print("shape:")
print(df_consolidado.shape)

print("\ncolunas:")
print(df_consolidado.columns.tolist())

shape:
(4000, 11)

colunas:
['ticket_id', 'cliente_id', 'data_abertura', 'canal', 'categoria', 'prioridade', 'tempo_resposta_horas', 'tempo_resolucao_horas', 'satisfacao_cliente', 'valor_plano', 'segmento']


##**EXERCÍCIO 9**

###9.1 Agrupe o DataFrame consolidado por categoria usando groupby.

In [67]:
grupos_categoria = df_consolidado.groupby("categoria")

### 9.2 Calcule, para cada grupo, a média de tempo_resposta_horas e a contagem de tickets.

In [68]:
resumo_categoria = grupos_categoria.agg(
    tempo_resposta_medio=("tempo_resposta_horas", "mean"),
    total_tickets=("ticket_id", "count"),
)
print(resumo_categoria)

               tempo_resposta_medio  total_tickets
categoria                                         
bug                        3.741420           1387
cobranca                   3.928200           1028
duvida_de_uso              3.717701           1192
elogio                     4.031247            393


### 9.3 Ordene o resultado da categoria com maior tempo médio de resposta para a menor e aponte, em uma frase, qual categoria merece atenção prioritária do time de suporte.

In [69]:
resumo_categoria = resumo_categoria.sort_values("tempo_resposta_medio", ascending=False)
print(resumo_categoria)

               tempo_resposta_medio  total_tickets
categoria                                         
elogio                     4.031247            393
cobranca                   3.928200           1028
bug                        3.741420           1387
duvida_de_uso              3.717701           1192


Considerando apenas o tempo de resposta, *elogio* aparece em primeiro lugar, porém como a diferença é pequena para as demais categorias e não apresenta risco operacional, a prioridade seria *cobranca*, já que soma o segundo maior tempo médio a um volume expressivo de 1028 *tickets*, com potencial impacto financeiro direto em caso de atraso.